### Tools 

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code 

In [40]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] =  os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.1-8b-instant")
response = model.invoke("Write few words about me")
response

AIMessage(content="Unfortunately, I don't have any information about you. You can tell me your name, interests, or hobbies, and I'd be happy to write a few words about you.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 40, 'total_tokens': 77, 'completion_time': 0.057014171, 'completion_tokens_details': None, 'prompt_time': 0.001819595, 'prompt_tokens_details': None, 'queue_time': 0.047390964, 'total_time': 0.058833766}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d35f5-a18c-7172-b25b-0db067a71b65-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 37, 'total_tokens': 77})

In [41]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at the location """
    return f"It's sunny in {location}"


model_wtih_tools = model.bind_tools([get_weather]) # we can add list of tools

In [42]:
response = model_wtih_tools.invoke("What is the weather in Mumbai")
print(response)


content='' additional_kwargs={'tool_calls': [{'id': 'ybvgs9q28', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 218, 'total_tokens': 233, 'completion_time': 0.024459338, 'completion_tokens_details': None, 'prompt_time': 0.014159237, 'prompt_tokens_details': None, 'queue_time': 0.047759183, 'total_time': 0.038618575}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d35f5-a3bc-7072-97f4-22feb56a7e02-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'ybvgs9q28', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 218, 'output_tokens': 15, 'total_tokens': 233}


In [43]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Mumbai'},
  'id': 'ybvgs9q28',
  'type': 'tool_call'}]

### Tool execution loop

In [44]:
 # Step 1: Model genrates tool calls

 messages = [{"role": "user", "content": "what's the weather in Mumbai?"}]
 ai_msg = model_wtih_tools.invoke(messages)
 messages.append(ai_msg)

 # Step 2: Execute the tools and collect results
 for tool_call in ai_msg.tool_calls:
    # Executes the tool with the generated arguments
    tool_results = get_weather.invoke(tool_call)
    messages.append(tool_results)

 # Step 3: Pass results back to model for final response
 final_response = model_wtih_tools.invoke(messages)
 print(final_response.text)

Note: The response 'It's sunny in Mumbai' is a generic answer, it's not based on the output of the get_weather function. To get the actual output of the function, you should replace 'It's sunny in Mumbai' with 'The weather in Mumbai is: ' + result, where result is the output of the get_weather function.

To get the actual output, you would need to call the get_weather function and store its output in a variable, like this:

result = get_weather(location="Mumbai")
print('The weather in Mumbai is: ' + result)

However, since the get_weather function is not defined in this environment, the above code will not work as expected.


In [45]:
messages

[{'role': 'user', 'content': "what's the weather in Mumbai?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'nkxq62ty1', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 219, 'total_tokens': 234, 'completion_time': 0.025208854, 'completion_tokens_details': None, 'prompt_time': 0.012935761, 'prompt_tokens_details': None, 'queue_time': 0.045226549, 'total_time': 0.038144615}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d35f5-a4eb-7082-b790-05829109bc49-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'nkxq62ty1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 15, 'total_tokens': 234}),
 ToolMessage(content="It's su